# VirtualMuni - Agente Inteligente con RAG (Colab rápido)
## Municipalidad de Puente Alto

Este notebook implementa el **agente con RAG** de forma RÁPIDA usando **Groq**
(API gratuita de LLM en la nube; modelos responden en segundos).

> **Antes de empezar (1 minuto):** obtén una clave API gratuita en
> https://console.groq.com -> crea cuenta / inicia sesión -> *API Keys* ->
> *Create API Key* -> copia la clave. Luego pégala en la celda de abajo.

## 1. Instalar dependencias (tarda ~1-2 min)

In [ ]:
!pip install -q groq langchain-groq langchain-huggingface sentence-transformers chromadb duckduckgo-search

## 2. Configuración de la API (Groq) y datos internos

In [ ]:
import os
from pathlib import Path
from getpass import getpass

# --- 1) Intentar cargar la clave desde el archivo .env local (proyecto VSCode) ---
def _leer_env_clave(candidatos):
    for f in candidatos:
        try:
            if f.exists():
                for linea in f.read_text(encoding='utf-8').splitlines():
                    linea = linea.strip()
                    if linea.startswith('GROQ_API_KEY=') and not linea.startswith('#'):
                        return linea.split('=', 1)[1].strip()
        except Exception:
            continue
    return ''

candidatos = [Path.cwd() / '.env', Path.cwd().parent / '.env', Path('/content/.env')]
clave = os.environ.get('GROQ_API_KEY', '') or _leer_env_clave(candidatos)

# --- 2) Si no hay clave, pedirla ---
if not clave:
    clave = getpass('Pega tu GROQ API KEY (console.groq.com): ')

os.environ['GROQ_API_KEY'] = clave
print('API key configurada correctamente.')

# --- Carpeta de documentos municipales (fuente del RAG) ---
DOCS_DIR = Path('/content/muni_docs')
DOCS_DIR.mkdir(exist_ok=True)

# --- Registro simulado de tramites para las automatizaciones ---
TRAMITES = [
    {'folio': 'PA-2025-0001', 'tipo': 'Certificado de Nacimiento', 'estado': 'En revision'},
    {'folio': 'PA-2025-0002', 'tipo': 'Permiso de Circulacion', 'estado': 'Aprobado'},
    {'folio': 'PA-2025-0003', 'tipo': 'Patente Comercial', 'estado': 'Pendiente de documentacion'},
]
print('Registro de tramites simulado listo:', len(TRAMITES))

In [ ]:
# --- Crear/actualizar los documentos municipales (fuente del RAG) ---
documentos = {
    'certificado_nacimiento.txt': 'MANUAL DE TRAMITES MUNICIPALES - CERTIFICADO DE NACIMIENTO. El certificado de nacimiento es emitido por el Registro Civil del municipio. Requisitos: cedula de identidad del padre, madre o apoderado. Para mayores de 18 anos, solo su cedula vigente. Valor: sin costo para nacimientos del mismo ano; $3.500 CLP para anos anteriores. Horario: lunes a viernes de 08:30 a 14:00. Oficina: Registro Civil, segundo piso, modulo 3. Tambien disponible en linea con firma electronica (Ley 19.799).',
    'permiso_circulacion.txt': 'MANUAL DE TRAMITES MUNICIPALES - PERMISO DE CIRCULACION. Se renueva anualmente entre el 1 de febrero y el 31 de marzo. Requisitos: avaluacion fiscal, SOAP vigente, padron anterior, cedula de identidad. Descuento 5% si se paga en linea durante los primeros 15 dias de febrero. Se paga en un solo pago. Multa y recargo por atraso.',
    'patente_comercial.txt': 'MANUAL DE TRAMITES MUNICIPALES - PATENTE COMERCIAL. Permiso obligatorio para actividades comerciales, industriales o de servicios. Requisitos: iniciacion de actividades en SII, certificado de informaciones previas, informe sanitario (alimentos), cedula del solicitante. Resolucion en 15 a 30 dias habiles. Se renueva anualmente en enero. No se puede operar sin la patente definitiva.',
    'horarios.txt': 'GUIA DE ATENCION. Edificio consistorial: Avenida Concha y Toro 820, Puente Alto. Horario: lunes a jueves 08:30-14:00 y 15:00-16:30. Viernes 08:30-14:00. Registro Civil: segundo piso, modulo 3, lunes a viernes 08:30-14:00. Unidad de Patentes: segundo piso, modulo 5. Tramites en linea: tercer piso, modulo 7.',
}
if not any(DOCS_DIR.glob('*.txt')):
    for nombre, contenido in documentos.items():
        (DOCS_DIR / nombre).write_text(contenido, encoding='utf-8')
    print('Documentos internos creados:', len(documentos))
else:
    print('Documentos internos ya existentes.')

print('Fuente interna del RAG lista.')

## 3. Pipeline RAG: indexar documentos en ChromaDB

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Cargar documentos
docs = []
for p in DOCS_DIR.glob('*.txt'):
    loader = TextLoader(str(p), encoding='utf-8')
    for d in loader.load():
        d.metadata['source'] = p.name
        docs.append(d)

# Dividir en fragmentos (chunking)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f'Fragmentos: {len(chunks)}')

# Embeddings multilingües gratuitos
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Indexar en ChromaDB
vectorstore = Chroma.from_documents(chunks, embeddings)
print('Índice vectorial creado correctamente.')

## 4. LLM rápido con Groq (responde en segundos)

In [ ]:
from langchain_groq import ChatGroq

# Modelo gratuito de Groq (rápido y sin costo)
llm = ChatGroq(
    model='openai/gpt-oss-120b',   # potente y gratuito en Groq; alternativas: 'openai/gpt-oss-20b'
    temperature=0.2,
    api_key=os.environ['GROQ_API_KEY'],
)

# Quick check
resp = llm.invoke('Di solo: listo')
print('Conexión con Groq OK ->', resp.content)


## 5. Prompt del sistema (rol, reglas y trazabilidad)

In [ ]:
SYSTEM_PROMPT = '''
Eres "VirtualMuni", asistente virtual oficial de la Municipalidad de Puente Alto.
- Respondes en español, de forma clara y amable.
- Los datos de TRÁMITES (requisitos, valores, horarios, ubicaciones) debes
  obtenerlos EXCLUSIVAMENTE de los documentos de contexto entregados.
- NUNCA inventes requisitos ni valores.
- Si la información no está en el contexto, responde que no dispones de ella
  y sugiere contactar a la municipalidad.
- Al final de cada respuesta sobre trámites, indica la fuente entre corchetes,
  por ejemplo: [Fuente: certificado_nacimiento.txt]
- Usa viñetas para enumerar requisitos o pasos.
'''
print('Prompt del sistema definido.')

## 6. Demostración del pipeline RAG (recuperar + generar)

In [ ]:
def recuperar_docs(consulta, k=3):
    docs = vectorstore.similarity_search(consulta, k=k)
    return '\n\n'.join(f'[Fuente: {d.metadata["source"]}]\n{d.page_content}' for d in docs)

def responder_rag(pregunta):
    contexto = recuperar_docs(pregunta)
    mensaje = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'DOCUMENTOS DE CONTEXTO:\n{contexto}\n\nConsulta del ciudadano:\n{pregunta}'},
    ]
    return llm.invoke(mensaje).content

# --- Pruebas RAG ---
preguntas = [
    '¿Qué necesito para obtener un certificado de nacimiento?',
    '¿Cuál es el horario de atención de la municipalidad?  ',
    '¿Cuánto cuesta el permiso de circulación y cuándo debo renovarlo?',
]

print('=== DEMOSTRACIÓN PIPELINE RAG ===\n')
for q in preguntas:
    print('>> Pregunta:', q)
    try:
        r = responder_rag(q)
        print('>> Respuesta:', r, '\n')
    except Exception as e:
        print('>> ERROR:', e, '\n')

## 7. Herramientas de automatización (agendar cita, consultar estado, generar solicitud)

In [ ]:
from datetime import datetime

def agendar_cita(nombre, tramite, fecha, hora):
    folio = f"CITA-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    return (f"Cita agendada correctamente. Folio: {folio}. Trámite: {tramite} "
            f"para {nombre} el {fecha} a las {hora}.")

def consultar_estado(folio):
    for t in TRAMITES:
        if t['folio'].lower() == folio.strip().lower():
            return f"Trámite {t['folio']} ({t['tipo']}): estado actual = {t['estado']}."
    return f"No se encontró un trámite con el folio '{folio}'. Verifica el folio o contacta a la mesa de ayuda."

def generar_solicitud(tipo_tramite, nombre, rut):
    archivo = f"solicitud_{tipo_tramite.replace(' ', '_')}.txt"
    contenido = (f"SOLICITUD DE TRÁMITE - MUNICIPALIDAD DE PUENTE ALTO\n"
                 f"Tipo de trámite: {tipo_tramite}\nSolicitante: {nombre}\n"
                 f"RUT: {rut}\nFecha: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    Path(archivo).write_text(contenido, encoding='utf-8')
    return f"Solicitud generada y guardada como '{archivo}'. Puedes descargarla desde el panel de archivos de Colab."

print('=== PRUEBA DE AUTOMATIZACIONES ===')
print('\n[1] Agendar cita:')
print(agendar_cita('Ana Pérez', 'Certificado de Nacimiento', '2025-06-10', '10:30'))
print('\n[2] Consultar estado (folio PA-2025-0002):')
print(consultar_estado('PA-2025-0002'))
print('\n[3] Consultar estado (folio inexistente PA-9999):')
print(consultar_estado('PA-9999'))
print('\n[4] Generar solicitud:')
print(generar_solicitud('Permiso de Circulación', 'Luis Gómez', '12.345.678-9'))

## 8. Agente con control de contexto (memoria conversacional)

In [ ]:
# Memoria conversacional: mantiene coherencia entre preguntas
historial = []

def chat_municipal(pregunta):
    contexto = recuperar_docs(pregunta)
    historial_texto = '\n'.join(f'Ciudadano: {h["q"]}\nVirtualMuni: {h["a"]}' for h in historial[-4:])
    mensaje = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': (
                f'DOCUMENTOS DE CONTEXTO:\n{contexto}\n\n'
                f'HISTORIAL DE LA CONVERSACIÓN:\n{historial_texto}\n\n'
                f'Consulta del ciudadano: {pregunta}'
            ),
        },
    ]
    respuesta = llm.invoke(mensaje).content
    historial.append({'q': pregunta, 'a': respuesta})
    return respuesta

# --- Prueba de conversación con contexto ---
print('== P1 ==')
print(chat_municipal('¿Qué necesito para un certificado de nacimiento?'))
print('\n== P2 (sigue la conversación) ==')
print(chat_municipal('¿Y dónde lo tramito?'))
print('\nHistorial almacenado:', len(historial), 'turnos')

## 9. Recuperación externa (búsqueda web) con DuckDuckGo

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

buscar_web = DuckDuckGoSearchRun()

consulta_externa = 'Puente Alto clima hoy'
print('=== BÚSQUEDA WEB EXTERNA ===')
print('Consulta:', consulta_externa)
try:
    resultado = buscar_web.run(consulta_externa)
    print('Resultado (primeros 500 caracteres):\n', resultado[:500])
except Exception as e:
    print('La búsqueda web requiere conexión; error:', e)

## 10. Agente completo (ReAct) con todas las herramientas

In [ ]:
from langchain_community.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate

herramientas = [
    Tool(name='recuperar_documentos', func=recuperar_docs,
         description='Recupera información de documentos municipales internos. Para trámites, requisitos, horarios y valores.'),
    Tool(name='buscar_web', func=buscar_web.run,
         description='Busca información pública actualizada (clima, noticias). Solo información externa.'),
    Tool(name='agendar_cita', func=agendar_cita,
         description='Agenda una cita municipal. Argumentos: nombre, trámite, fecha YYYY-MM-DD, hora HH:MM.'),
    Tool(name='consultar_estado', func=consultar_estado,
         description='Consulta el estado de un trámite por folio. Argumento: folio.'),
    Tool(name='generar_solicitud', func=generar_solicitud,
         description='Genera un formulario de solicitud. Argumentos: tipo_tramite, nombre, rut.'),
]

prompt_agente = PromptTemplate.from_template(
    """Eres VirtualMuni, asistente de la Municipalidad de Puente Alto.
Responde en español. Decide qué herramienta usar según la consulta:
- Datos de trámites -> recuperar_documentos
- Información externa -> buscar_web
- Acción (agendar, consultar estado, generar solicitud) -> la herramienta correspondiente
No inventes información.
Consulta: {input}\n
Herramientas disponibles: {tools}\n
Nombres de herramientas: {tool_names}\n
Paso a paso: {agent_scratchpad}"""
)

agente = create_react_agent(llm, herramientas, prompt_agente)
executor = AgentExecutor(agent=agente, tools=herramientas, verbose=True, handle_parsing_errors=True, max_iterations=5)

# Prueba 1: consulta de trámite (debería usar RAG)
print('=== AGENTE: consulta de trámite ===')
try:
    print(executor.invoke({'input': 'Necesito sacar un certificado de nacimiento, ¿qué requisitos me piden?'}))
except Exception as e:
    print('Error:', e)

# Prueba 2: acción (debería agendar cita)
print('\n=== AGENTE: agendar cita ===')
try:
    print(executor.invoke({'input': 'Quiero pedir hora para el permiso de circulación el 20 de febrero a las 09:00, soy Luis Gómez'}))
except Exception as e:
    print('Error:', e)

## Listo

Ya tienes el agente **VirtualMuni** funcionando: RAG (recupera de documentos
internos), búsqueda web externa y automatizaciones. Si Groq no respondiera,
revisa que la GROQ_API_KEY sea correcta y que la cuenta tenga créditos gratuitos
activos (los modelos de Groq son actualmente gratuitos).